# Module 05 — Lab: Advanced API features

Cache, batch, thinking, structured outputs.

In [ ]:
import os, time, json
from dotenv import load_dotenv
from anthropic import Anthropic
from pydantic import BaseModel, Field

load_dotenv('../.env')
client = Anthropic()
MODEL = os.getenv('ANTHROPIC_MODEL', 'claude-sonnet-4-6')

## 1. Prompt caching

In [ ]:
BIG_SYSTEM = (
    'You are a meticulous code reviewer for a Python codebase. '
    'You always answer in three sections: Findings, Severity, Suggested Fix. '
    + '\n\nStyle reference: ' + ('keep diffs minimal. ' * 200)
)

def review(snippet, label):
    t0 = time.perf_counter()
    r = client.messages.create(
        model=MODEL, max_tokens=300,
        system=[{'type':'text','text': BIG_SYSTEM, 'cache_control':{'type':'ephemeral'}}],
        messages=[{'role':'user','content': f'<code>\n{snippet}\n</code>'}],
    )
    dt = time.perf_counter() - t0
    u = r.usage
    print(f'{label:<12} {dt*1000:6.0f}ms  '
          f'in={u.input_tokens:<4} write={u.cache_creation_input_tokens:<4} read={u.cache_read_input_tokens:<4}')

review('def add(a,b): return a+b', '1st call')
review('def mul(a,b): return a*b', '2nd call')
review('def div(a,b): return a/b', '3rd call')

## 2. Cache invalidation

In [ ]:
# Change one character in the prefix
BIG_SYSTEM_2 = BIG_SYSTEM + '!'
r = client.messages.create(
    model=MODEL, max_tokens=200,
    system=[{'type':'text','text': BIG_SYSTEM_2, 'cache_control':{'type':'ephemeral'}}],
    messages=[{'role':'user','content': 'def sub(a,b): return a-b'}],
)
u = r.usage
print(f'invalidated -> write={u.cache_creation_input_tokens} read={u.cache_read_input_tokens}')

## 3. Extended thinking vs not

In [ ]:
PROBLEM = 'How many trailing zeros does 100! have? Answer with just the integer.'

t0 = time.perf_counter()
plain = client.messages.create(
    model=MODEL, max_tokens=128,
    messages=[{'role':'user','content':PROBLEM}],
)
plain_dt = time.perf_counter() - t0

t0 = time.perf_counter()
thinking = client.messages.create(
    model=MODEL, max_tokens=4096,
    thinking={'type':'enabled','budget_tokens':3000},
    messages=[{'role':'user','content':PROBLEM}],
)
think_dt = time.perf_counter() - t0

def final_text(r):
    return ''.join(b.text for b in r.content if b.type == 'text')

print(f'plain    {plain_dt*1000:.0f}ms  out_tokens={plain.usage.output_tokens}  answer: {final_text(plain)}')
print(f'thinking {think_dt*1000:.0f}ms  out_tokens={thinking.usage.output_tokens}  answer: {final_text(thinking)}')

## 4. Pydantic + forced tool

In [ ]:
class ActionItem(BaseModel):
    owner: str
    task: str
    due: str = Field(description='YYYY-MM-DD')

class MeetingNotes(BaseModel):
    attendees: list[str]
    decisions: list[str]
    action_items: list[ActionItem]

TRANSCRIPT = '''Maya, Raj, and Priya met today.
We decided to deprecate the v1 API by end of Q3.
Raj will draft the migration guide by 2026-06-15.
Priya owns customer comms, due 2026-06-20.'''

tool = {
    'name': 'save_meeting_notes',
    'description': 'Save structured meeting notes.',
    'input_schema': MeetingNotes.model_json_schema(),
}

r = client.messages.create(
    model=MODEL, max_tokens=1024,
    tools=[tool],
    tool_choice={'type':'tool','name':'save_meeting_notes'},
    messages=[{'role':'user','content': TRANSCRIPT}],
)
raw = next(b.input for b in r.content if b.type == 'tool_use')
notes = MeetingNotes(**raw)
print(notes.model_dump_json(indent=2))

## 5. Tiny batch (optional — incurs cost; comment out if not needed)

In [ ]:
rows = ['Translate to French: hello', 'Translate to French: goodbye', 'Translate to French: thank you']
batch = client.messages.batches.create(
    requests=[
        {'custom_id': f'row-{i}',
         'params': {'model': MODEL, 'max_tokens': 64,
                    'messages':[{'role':'user','content': r}]}}
        for i, r in enumerate(rows)
    ],
)
print('batch id:', batch.id, 'status:', batch.processing_status)

while True:
    b = client.messages.batches.retrieve(batch.id)
    print('  ...', b.processing_status)
    if b.processing_status == 'ended':
        break
    time.sleep(5)

for result in client.messages.batches.results(batch.id):
    msg = result.result.message
    print(result.custom_id, '->', msg.content[0].text)